# SMOTE + Noise v6 — Aleatoric Uncertainty + Vine Copula + Adaptive Noise

## So sánh kết quả qua các phiên bản

| Chỉ số | v3 | v4 | v5 | **v6 (mới)** |
|---|---|---|---|---|
| Frobenius Spearman | 1.11 | 0.84 | 1.08 | **TBD** |
| Frobenius Pearson  | 2.34 | 1.44 | 1.46 | **TBD** |
| Adversarial Accuracy | — | — | 0.907 | **target ≈ 0.75** |
| Wasserstein total | — | — | 0.491 | **target < 0.45** |

## Cải tiến cốt lõi trong v6

1. **Adaptive Noise** — `σ = α × (1 − MI_norm) × local_std` thay vì Gaussian noise toàn cục `NOISE_LEVEL=0.015`
   - Biến có MI cao (Avg Temp) → noise nhỏ (giữ tín hiệu)
   - Biến có MI thấp (Bulk_Density) → noise lớn hơn (tái tạo aleatoric uncertainty)
   - local_std tính per-crop để phản ánh microclimate variability thực tế

2. **Gaussian Copula per cluster** — thay thế Clustered Iman-Conover bằng Gaussian Copula
   - IC chỉ giữ rank correlation (Spearman)
   - Copula giữ được toàn bộ joint distribution bao gồm tail dependence
   - Áp dụng per cluster (giữ kiến trúc cluster của v4/v5)

3. **Probabilistic Yield constraint** — thay hard rejection bằng quantile bound per-crop
   - `Yield_syn ∈ [Q0.025(crop), Q0.975(crop)]` tính từ dữ liệu gốc
   - Giữ được aleatoric uncertainty thật (dao động do sâu bệnh, giống cây)
   - Không làm dataset quá deterministic

4. **District-level cols inheritance** — `Bulk_Density` có đúng 1 giá trị/district
   - Đây là thuộc tính địa lý, không phải biến sinh học
   - SMOTE nội suy tạo ra 19,982 giá trị continuous → classifier phân biệt trivially
   - v6: kế thừa từ parent sample (giống CATEGORICAL_COLS) → giữ đúng 13 giá trị discrete

5. **Adversarial Accuracy metric** — đánh giá synthetic data bằng XGBoost classifier
   - `accuracy ≈ 0.50` → synthetic indistinguishable from real
   - v5 baseline: 0.907 (quá dễ phân biệt)

## Kiến trúc giữ nguyên từ v5
- SEED=42, NUM_ROWS=20000, K_NEIGHBORS=5, N_CLUSTERS=5
- Log transform cho Area, Production (skewness ~8)
- Stratified SMOTE per Crop Name
- Tất cả hậu xử lý logic (Cell 5): mùa vụ, rainfall clip, domain bounds, flags, soil sum
- Tái tính derived cols (Cell 6): CN_Ratio, NDVI_Std, NDVI_CV, Rain_Temp_Ratio
- Categorical sync (Cell 7): Water_Availability_Cat fix, pH_Suitability, Dominant_Soil_Texture
- Tất cả FIX v5: Heat_Stress_Days, Nitrogen/OC bounds, NDVI_Std precision


In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from scipy.stats import rankdata, norm, spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import wasserstein_distance

# ============================================================
# 0. CONFIG  — giữ nguyên từ v5
# ============================================================
SEED        = 42
NUM_ROWS    = 20000
K_NEIGHBORS = 5
N_CLUSTERS  = 5      # số cluster cho Copula (giữ nguyên v5)
NOISE_ALPHA = 0.08   # hệ số scale cho adaptive noise (thay NOISE_LEVEL=0.015)
np.random.seed(SEED)

# ============================================================
# 1. LOAD DATA  — giữ nguyên từ v5
# ============================================================
print('[1/9] Đọc dữ liệu...')
data = pd.read_csv('Agri_Data_Cleaned.csv')
print(f'     Gốc: {data.shape[0]:,} dòng × {data.shape[1]} cột')

CATEGORICAL_COLS = [
    'District', 'Season', 'Crop Name', 'Transplant',
    'Growth', 'Harvest', 'pH_Suitability',
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max'
]
DERIVED_COLS = [
    'CN_Ratio', 'Rain_Temp_Ratio',
    'Rootzone_Surface_Diff', 'Moisture_Ratio',
    'NDVI_Season_Range', 'NDVI_Season_CV', 'NDVI_Season_Std'
]
BASE_NUMERIC_COLS = [
    c for c in data.columns
    if c not in CATEGORICAL_COLS + DERIVED_COLS and c != 'Yield'
]
LOG_COLS = ['Area', 'Production']   # skewness ~8 → log transform (giữ v5)

# v6: Bulk_Density có đúng 1 giá trị per District (thuộc tính địa lý, không phải sinh học)
# → không nên nội suy SMOTE → kế thừa từ parent sample (như CATEGORICAL_COLS)
DISTRICT_LEVEL_COLS = ['Bulk_Density']

IC_COLS  = [c for c in BASE_NUMERIC_COLS if c not in LOG_COLS + DISTRICT_LEVEL_COLS]

print(f'     Base numeric: {len(BASE_NUMERIC_COLS)} | Copula cols: {len(IC_COLS)} | Log cols: {LOG_COLS}')


[1/9] Đọc dữ liệu...
     Gốc: 4,178 dòng × 51 cột
     Base numeric: 30 | Copula cols: 27 | Log cols: ['Area', 'Production']


In [2]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def nearest_psd(A, eps=1e-8):
    """Ép matrix về Positive Semi-Definite — giữ nguyên từ v5"""
    ev, evec = np.linalg.eigh(A)
    ev = np.maximum(ev, eps)
    r  = evec @ np.diag(ev) @ evec.T
    np.fill_diagonal(r, 1.0)
    return r


def compute_adaptive_noise_std(orig_df, base_cols, alpha=0.035):
    """
    Adaptive Noise — CẢI TIẾN #1 của v6.

    v5 dùng noise toàn cục: noise = N(0, 0.015) cho mọi biến
    → Bulk_Density (MI≈0) bị smooth quá → classifier phân biệt dễ dàng
    → Avg Temp (MI≈1) bị noise làm mờ tín hiệu

    v6: σ_col = alpha × (1 − MI_norm_col) × local_std_col
    → Biến MI cao (Avg Temp): σ nhỏ → giữ tín hiệu nông học
    → Biến MI thấp (Bulk_Density): σ lớn → tái tạo aleatoric uncertainty thực
    → local_std per-crop phản ánh microclimate/soil variability địa phương
    """
    target = 'Yield'
    feat_cols = [c for c in base_cols if c in orig_df.columns and c != target]

    # MI với Yield
    X = orig_df[feat_cols].fillna(orig_df[feat_cols].median())
    y = orig_df[target]
    mi_raw  = mutual_info_regression(X, y, random_state=SEED)
    mi_norm = pd.Series(mi_raw, index=feat_cols)
    mi_norm = (mi_norm - mi_norm.min()) / (mi_norm.max() - mi_norm.min() + 1e-10)

    # Local std per-crop (trong không gian scaled [0,1])
    scaler = MinMaxScaler()
    scaled = pd.DataFrame(
        scaler.fit_transform(orig_df[feat_cols].fillna(orig_df[feat_cols].median())),
        columns=feat_cols
    )
    scaled['Crop Name'] = orig_df['Crop Name'].values
    global_std = scaled[feat_cols].std()
    local_std  = scaled.groupby('Crop Name')[feat_cols].std().mean()  # mean over crops
    local_std  = local_std.fillna(global_std)
    # v6: dùng max(local_std, global_std*0.3) để tránh noise quá nhỏ với quasi-discrete vars
    local_std  = local_std.combine(global_std * 0.3, max)
    local_std  = local_std.clip(lower=0.01)   # floor để tránh noise=0

    # σ = alpha × (1 - MI_norm) × local_std
    noise_std = alpha * (1.0 - mi_norm) * local_std
    noise_std = noise_std.clip(lower=0.005, upper=0.12)  # sàn/trần an toàn

    print(f'     Adaptive noise σ — min: {noise_std.min():.4f}, max: {noise_std.max():.4f}')
    print(f'     Top 3 noise cao: {noise_std.sort_values(ascending=False).head(3).to_dict()}')
    print(f'     Top 3 noise thấp: {noise_std.sort_values().head(3).to_dict()}')
    return noise_std


def gaussian_copula_per_cluster(syn_df, orig_df, cols, n_clusters=5):
    """
    Gaussian Copula per Cluster — CẢI TIẾN #2 của v6.

    v5 dùng Clustered Iman-Conover:
    → Chỉ khớp Spearman rank correlation
    → Không mô hình được tail dependence

    v6 dùng Gaussian Copula trong từng cluster:
    Pipeline:
      1. Chuyển marginals → uniform [0,1] qua empirical CDF
      2. Chuyển uniform → Normal qua probit (Φ⁻¹)
      3. Khớp correlation matrix trong không gian Normal (Gaussian Copula)
      4. Sinh mẫu mới từ Gaussian Copula
      5. Inverse CDF → trả về không gian gốc

    Giữ kiến trúc cluster giống v5 (hierarchical clustering trên correlation distance).
    """
    valid = [c for c in cols if c in syn_df.columns and c in orig_df.columns]

    # --- Bước 1: Phân cụm biến (giữ nguyên từ v5) ---
    corr_abs = orig_df[valid].corr('spearman').abs()
    dist_mat = squareform(np.clip(1 - corr_abs.values, 0, None))
    Z        = linkage(dist_mat, method='average')
    labels   = fcluster(Z, t=n_clusters, criterion='maxclust')

    clusters = {}
    for i, c in enumerate(valid):
        clusters.setdefault(labels[i], []).append(c)

    cluster_sizes = [len(v) for v in clusters.values()]
    print(f'     Clusters ({n_clusters}): {cluster_sizes} cols each')

    res = syn_df.copy()
    n   = len(res)

    for cl_id, cl_cols in clusters.items():
        if len(cl_cols) < 2:
            continue

        orig_vals = orig_df[cl_cols].values.astype(float)
        syn_vals  = res[cl_cols].values.astype(float)
        n_orig    = len(orig_vals)

        # --- Bước 2: Empirical CDF → Uniform ---
        # Dùng rank của syn_vals trên orig distribution
        def to_uniform(x_syn, x_orig):
            """Map x_syn vào [0,1] dùng empirical CDF của x_orig"""
            u = np.zeros_like(x_syn)
            x_sorted = np.sort(x_orig)
            for i in range(len(x_syn)):
                u[i] = (np.searchsorted(x_sorted, x_syn[i], side='right')) / (n_orig + 1)
            return np.clip(u, 1e-6, 1 - 1e-6)

        U = np.column_stack([to_uniform(syn_vals[:, j], orig_vals[:, j])
                             for j in range(len(cl_cols))])

        # --- Bước 3: Uniform → Normal (Probit) ---
        Z_scores = norm.ppf(U)  # shape (n, k)

        # --- Bước 4: Gaussian Copula — khớp correlation từ orig ---
        # Compute target copula correlation từ orig
        U_orig = np.column_stack([
            np.clip(rankdata(orig_vals[:, j]) / (n_orig + 1), 1e-6, 1 - 1e-6)
            for j in range(len(cl_cols))
        ])
        Z_orig = norm.ppf(U_orig)
        corr_target = nearest_psd(np.corrcoef(Z_orig.T))

        # Correlation của Z_scores hiện tại
        corr_current = nearest_psd(np.corrcoef(Z_scores.T))

        try:
            L_target  = np.linalg.cholesky(corr_target)
            L_current = np.linalg.cholesky(corr_current)
        except np.linalg.LinAlgError:
            continue

        # Transform Z_scores: decorrelate hiện tại → apply target correlation
        Z_white       = Z_scores @ np.linalg.inv(L_current).T
        Z_transformed = Z_white @ L_target.T

        # --- Bước 5: Rank rearrangement (giữ marginals từ SMOTE) ---
        for j, col in enumerate(cl_cols):
            target_ranks = rankdata(Z_transformed[:, j], 'ordinal').astype(int) - 1
            res[col]     = np.sort(res[col].values)[target_ranks]

    return res


def yield_probabilistic_constraint(syn_df, orig_df, q_low=0.01, q_high=0.99):
    """
    Probabilistic Yield Constraint — CẢI TIẾN #3 của v6.

    v5: Yield = Production/Area, không có kiểm tra tính hợp lý per-crop
    → Yield có thể ra giá trị phi thực tế (Production/Area từ 2 samples khác nhau)

    v6: Sau khi tính Yield = Production/Area, kiểm tra xem Yield_syn
    có nằm trong khoảng [Q_low, Q_high] của crop đó không.
    Nếu không → clamp về bound gần nhất.

    Giữ được aleatoric uncertainty (không hard-reject) nhưng loại bỏ extreme outlier phi thực tế.
    Bounds tính từ Q0.01 đến Q0.99 (rộng hơn Q0.025/Q0.975 để giữ tính đa dạng).
    """
    crop_bounds = orig_df.groupby('Crop Name')['Yield'].agg(
        lo=lambda x: x.quantile(q_low),
        hi=lambda x: x.quantile(q_high)
    ).reset_index()

    syn_df = syn_df.merge(crop_bounds, on='Crop Name', how='left')

    # Clamp thay vì reject — giữ aleatoric uncertainty
    before = syn_df['Yield'].copy()
    syn_df['Yield'] = syn_df['Yield'].clip(lower=syn_df['lo'], upper=syn_df['hi'])
    n_clamped = (syn_df['Yield'] != before).sum()
    print(f'     Yield clamped: {n_clamped} / {len(syn_df)} ({n_clamped/len(syn_df)*100:.1f}%)')

    syn_df.drop(columns=['lo', 'hi'], inplace=True)
    return syn_df


print('[Helpers] Loaded.')


[Helpers] Loaded.


In [3]:
# ============================================================
# 2b. TÍNH ADAPTIVE NOISE WEIGHTS (trước SMOTE)
# ============================================================
print('[2b/9] Tính Adaptive Noise weights...')
noise_std_map = compute_adaptive_noise_std(
    data, BASE_NUMERIC_COLS + ['Yield'], alpha=NOISE_ALPHA
)


[2b/9] Tính Adaptive Noise weights...
     Adaptive noise σ — min: 0.0050, max: 0.0206
     Top 3 noise cao: {'Nitrogen': 0.020647866437433843, 'Clay': 0.017983357775175703, 'Bulk_Density': 0.017917270779497826}
     Top 3 noise thấp: {'Area': 0.005, 'Avg Temp': 0.005, 'Avg Humidity': 0.005}


In [4]:
# ============================================================
# 3. STRATIFIED SMOTE + ADAPTIVE NOISE (per Crop)
# ============================================================
# Giữ nguyên toàn bộ kiến trúc SMOTE từ v5:
#   - Stratified per Crop Name
#   - Log transform cho Area, Production
#   - MinMaxScaler
#   - k=5 neighbors
#   - Parent categorical inheritance
# Chỉ thay: noise toàn cục → adaptive noise per-column
# ============================================================
print(f'[3/9] Stratified SMOTE + Adaptive Noise (k={K_NEIGHBORS})...')

crop_counts = data['Crop Name'].value_counts()
crop_target = (crop_counts / len(data) * NUM_ROWS).round().astype(int)
crop_target[crop_counts.index[0]] += NUM_ROWS - crop_target.sum()

all_synthetic = []
for crop_name, n_crop in crop_target.items():
    if n_crop <= 0:
        continue
    crop_df = data[data['Crop Name'] == crop_name].reset_index(drop=True)
    n_orig  = len(crop_df)

    if n_orig < 3:
        all_synthetic.append(crop_df.sample(n=n_crop, replace=True).reset_index(drop=True))
        continue

    data_num = crop_df[BASE_NUMERIC_COLS].fillna(crop_df[BASE_NUMERIC_COLS].median())
    data_log = data_num.copy()
    for col in LOG_COLS:                          # giữ log transform v5
        if col in data_log.columns:
            data_log[col] = np.log1p(data_log[col].clip(lower=0))

    scaler      = MinMaxScaler()
    data_scaled = scaler.fit_transform(data_log)

    k_actual = min(K_NEIGHBORS, n_orig - 1)
    nbrs     = NearestNeighbors(n_neighbors=k_actual + 1).fit(data_scaled)

    parent_idxs = np.random.choice(n_orig, n_crop, replace=True)
    parents     = data_scaled[parent_idxs]
    nb_idx      = nbrs.kneighbors(parents, return_distance=False)
    neigh_idxs  = np.array([np.random.choice(r[1:]) for r in nb_idx])
    neighbors   = data_scaled[neigh_idxs]

    ratios           = np.random.rand(n_crop, 1)
    synthetic_scaled = parents + ratios * (neighbors - parents)

    # === ADAPTIVE NOISE (thay np.random.normal(0, NOISE_LEVEL, ...)) ===
    # Bulk_Density sẽ bị drop và thay bằng parent value → không cần noise
    noise = np.zeros_like(synthetic_scaled)
    for j, col in enumerate(BASE_NUMERIC_COLS):
        if col in DISTRICT_LEVEL_COLS:
            continue   # skip — sẽ được thay bằng parent value
        col_sigma = noise_std_map.get(col, 0.015)   # fallback về v5 default
        noise[:, j] = np.random.normal(0, col_sigma, n_crop)
    synthetic_scaled = np.clip(synthetic_scaled + noise, 0, 1)
    # ===================================================================

    synthetic_num = scaler.inverse_transform(synthetic_scaled)
    syn_df_num    = pd.DataFrame(synthetic_num, columns=BASE_NUMERIC_COLS)
    for col in LOG_COLS:                          # giữ inverse log v5
        if col in syn_df_num.columns:
            syn_df_num[col] = np.expm1(syn_df_num[col]).clip(lower=0)

    parent_cats = crop_df.iloc[parent_idxs][CATEGORICAL_COLS].reset_index(drop=True)
    # v6: kế thừa Bulk_Density từ parent (district-level attribute — 1 giá trị/district)
    parent_dist_cols = crop_df.iloc[parent_idxs][DISTRICT_LEVEL_COLS].reset_index(drop=True)
    # Xóa Bulk_Density đã được SMOTE nội suy (sai), thay bằng giá trị từ parent
    syn_df_num = syn_df_num.drop(columns=DISTRICT_LEVEL_COLS, errors='ignore')
    all_synthetic.append(pd.concat([syn_df_num, parent_cats, parent_dist_cols], axis=1))

synthetic_data = pd.concat(all_synthetic, ignore_index=True)
print(f'     Sinh: {len(synthetic_data):,} mẫu từ {len(crop_target)} loại cây')


[3/9] Stratified SMOTE + Adaptive Noise (k=5)...
     Sinh: 20,000 mẫu từ 72 loại cây


In [5]:
# ============================================================
# 4. GAUSSIAN COPULA PER CLUSTER — BẢO TOÀN MULTIVARIATE STRUCTURE
# ============================================================
# Thay Clustered Iman-Conover (v5) bằng Gaussian Copula
# Giữ nguyên: N_CLUSTERS=5, hierarchical clustering trên correlation distance
# ============================================================
print(f'[4/9] Gaussian Copula per Cluster ({N_CLUSTERS} clusters)...')

synthetic_data = gaussian_copula_per_cluster(
    synthetic_data, data, IC_COLS, n_clusters=N_CLUSTERS
)
print('     Gaussian Copula hoàn tất.')


[4/9] Gaussian Copula per Cluster (5 clusters)...
     Clusters (5): [3, 3, 13, 6, 2] cols each
     Gaussian Copula hoàn tất.


In [6]:
# ============================================================
# 5. HẬU XỬ LÝ LOGIC  — giữ nguyên hoàn toàn từ v5
# ============================================================
print('[5/9] Hậu xử lý Logic...')

# --- A. ĐỒNG BỘ MÙA VỤ --- (v5)
cols_to_sync = ['Season', 'Transplant', 'Growth', 'Harvest']
synthetic_data = synthetic_data.drop(columns=cols_to_sync, errors='ignore')
sampled_rows = []
for crop in synthetic_data['Crop Name'].unique():
    idx = synthetic_data[synthetic_data['Crop Name'] == crop].index
    orig_subset = data[data['Crop Name'] == crop][cols_to_sync]
    if not orig_subset.empty:
        sampled_rows.append(orig_subset.sample(n=len(idx), replace=True).set_index(idx))
synthetic_data = pd.concat([synthetic_data, pd.concat(sampled_rows)], axis=1)

# --- B. RAINFALL CLIP (Percentile P5/P95) --- (v5)
geo  = data.groupby(['District','Season'])['Rainfall'].agg(
    R5=lambda x: x.quantile(0.05), R95=lambda x: x.quantile(0.95)).reset_index()
dist = data.groupby('District')['Rainfall'].agg(
    D5=lambda x: x.quantile(0.05), D95=lambda x: x.quantile(0.95)).reset_index()
synthetic_data = synthetic_data.merge(geo,  on=['District','Season'], how='left')
synthetic_data = synthetic_data.merge(dist, on='District', how='left')
g_min = data['Rainfall'].quantile(0.01)
g_max = data['Rainfall'].quantile(0.99)
synthetic_data['R5']  = synthetic_data['R5'].fillna(synthetic_data['D5']).fillna(g_min)
synthetic_data['R95'] = synthetic_data['R95'].fillna(synthetic_data['D95']).fillna(g_max)
synthetic_data['Rainfall'] = synthetic_data['Rainfall'].clip(
    lower=synthetic_data['R5'], upper=synthetic_data['R95']).round(2)
synthetic_data.drop(columns=['R5','R95','D5','D95'], inplace=True)

# --- C. CLIP ÂM & DOMAIN BOUNDS --- (v5 + FIX v5)
non_neg = ['Rainfall','Soil_Moisture_mm','Nitrogen','Organic_Carbon',
           'Wind_Max','Wind_Mean','Heat_Stress_Days',
           'sm_surface','sm_rootzone','EVI','LAI','FPAR',
           'Avg_Salinity_Index','Bulk_Density']
UPPER_BOUNDS = {
    'Nitrogen':        3.215,    # FIX v5
    'Organic_Carbon': 33.040,    # FIX v5
}
for col in non_neg:
    if col in synthetic_data.columns:
        lo = 0
        hi = UPPER_BOUNDS.get(col, None)
        synthetic_data[col] = synthetic_data[col].clip(lower=lo, upper=hi)

for col in [c for c in synthetic_data.columns if 'NDVI' in c and c in BASE_NUMERIC_COLS]:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)
if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)
for col in ['Min Relative Humidity','Avg Humidity','Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

def fix_min_mean_max(df, cmin, cmean, cmax):
    if {cmin, cmean, cmax}.issubset(df.columns):
        mask = df[cmin] > df[cmax]
        df.loc[mask, [cmin, cmax]] = df.loc[mask, [cmax, cmin]].values
        df[cmean] = df[cmean].clip(lower=df[cmin], upper=df[cmax])
    return df

synthetic_data = fix_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = fix_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = fix_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')
synthetic_data[['Min Temp','Avg Temp','Max Temp']] = synthetic_data[['Min Temp','Avg Temp','Max Temp']].round(1)

if {'Wind_Mean','Wind_Max'}.issubset(synthetic_data.columns):
    mask_w = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
    synthetic_data.loc[mask_w,'Wind_Mean'] = synthetic_data.loc[mask_w,'Wind_Max'] * 0.8

# --- D. SOIL SUM = 100% --- (v5)
soil_cols = ['Sand','Silt','Clay']
if all(c in synthetic_data.columns for c in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    total = synthetic_data[soil_cols].sum(axis=1).replace(0, 100)
    for c in soil_cols:
        synthetic_data[c] = (synthetic_data[c] / total * 100).round(2)
    synthetic_data['Clay'] = (100.0 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2).clip(lower=0)

# --- E. FLAGS 2 CHIỀU --- (v5)
WIND_FLAG_VAL  = float(data[data['is_extreme_Wind_Max']==1]['Wind_Max'].median())
WIND_NORM_MAX  = float(data[data['is_extreme_Wind_Max']==0]['Wind_Max'].max())
if 'is_extreme_Wind_Max' in synthetic_data.columns:
    m1 = (synthetic_data['is_extreme_Wind_Max']==1) & (synthetic_data['Wind_Max'] < WIND_FLAG_VAL)
    synthetic_data.loc[m1, 'Wind_Max'] = WIND_FLAG_VAL
    m0 = (synthetic_data['is_extreme_Wind_Max']==0) & (synthetic_data['Wind_Max'] >= WIND_FLAG_VAL)
    synthetic_data.loc[m0, 'Wind_Max'] = np.random.uniform(2.0, WIND_NORM_MAX, size=m0.sum())
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)

# FIX v5: Heat_Stress_Days
HEAT_EXTREME_VAL = 42.5
HEAT_NORMAL_MAX  = 41.0
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns:
    m1 = synthetic_data['is_extreme_Heat_Stress_Days'] == 1
    synthetic_data.loc[m1, 'Heat_Stress_Days'] = HEAT_EXTREME_VAL
    m0 = synthetic_data['is_extreme_Heat_Stress_Days'] == 0
    synthetic_data.loc[m0, 'Heat_Stress_Days'] = synthetic_data.loc[m0, 'Heat_Stress_Days'].clip(0, HEAT_NORMAL_MAX)

if {'Extreme_Heat_Risk','Is_Extreme_Heat'}.issubset(synthetic_data.columns):
    mc = (synthetic_data['Is_Extreme_Heat']==1) & (synthetic_data['Extreme_Heat_Risk']=='Low Risk')
    synthetic_data.loc[mc,'Extreme_Heat_Risk'] = 'High Risk'
    synthetic_data.loc[synthetic_data['Extreme_Heat_Risk']=='Low Risk','Is_Extreme_Heat'] = 0


[5/9] Hậu xử lý Logic...


In [7]:
# ============================================================
# 6. TÁI TÍNH DERIVED COLS  — giữ nguyên từ v5
# ============================================================
print('[6/9] Tái tính Derived cols...')

# Yield = Production / Area (v5)
synthetic_data['Area']       = synthetic_data['Area'].clip(lower=1.0)
synthetic_data['Production'] = synthetic_data['Production'].clip(lower=0.0)
synthetic_data['Yield']      = (synthetic_data['Production'] / synthetic_data['Area']).round(4)

# CN_Ratio (FIX v5)
if 'CN_Ratio' in data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (
        synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen']
    ).clip(6.960, 16.110).round(4)

# Rootzone_Surface_Diff (v5)
if 'Rootzone_Surface_Diff' in data.columns:
    synthetic_data['Rootzone_Surface_Diff'] = (
        synthetic_data['sm_rootzone'] - synthetic_data['sm_surface']).round(4)

# Moisture_Ratio (v5)
if 'Moisture_Ratio' in data.columns:
    sm_safe = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / sm_safe).round(4)

# NDVI_Season_Range (v5)
if 'NDVI_Season_Range' in data.columns:
    synthetic_data['NDVI_Season_Range'] = (
        synthetic_data['NDVI_Season_Max'] - synthetic_data['NDVI_Season_Min']).clip(0).round(4)

# NDVI_Season_Std (FIX v5 — clip với exact orig bounds)
if 'NDVI_Season_Std' in data.columns:
    RATIO_MEAN = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)).mean()
    RATIO_STD  = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)).std()
    noise_ratio = np.clip(
        np.random.normal(RATIO_MEAN, RATIO_STD, len(synthetic_data)), 0.40, 0.55)
    synthetic_data['NDVI_Season_Std'] = (
        synthetic_data['NDVI_Season_Range'] * noise_ratio
    ).clip(data['NDVI_Season_Std'].min(), data['NDVI_Season_Std'].max()).round(4)

# NDVI_Season_CV (FIX v5)
if 'NDVI_Season_CV' in data.columns:
    NDVI_CV_CAP   = 1.4848
    NDVI_MEAN_MIN = 0.1715
    mean_safe = synthetic_data['NDVI_Season_Mean'].abs().clip(lower=NDVI_MEAN_MIN)
    synthetic_data['NDVI_Season_CV'] = (
        synthetic_data['NDVI_Season_Std'] / mean_safe
    ).clip(upper=NDVI_CV_CAP).round(4)

# Rain_Temp_Ratio (v5)
if 'Rain_Temp_Ratio' in data.columns:
    ref_rtr  = data.groupby(['District','Season'])['Rain_Temp_Ratio'].median().reset_index()
    ref_rain = data.groupby(['District','Season'])['Rainfall'].median().reset_index()
    ref_rtr.columns  = ['District','Season','RTR_ref']
    ref_rain.columns = ['District','Season','Rain_ref']
    synthetic_data = synthetic_data.merge(ref_rtr,  on=['District','Season'], how='left')
    synthetic_data = synthetic_data.merge(ref_rain, on=['District','Season'], how='left')
    safe_ref = synthetic_data['Rain_ref'].replace(0, np.nan).fillna(1)
    synthetic_data['Rain_Temp_Ratio'] = (
        synthetic_data['RTR_ref'] * synthetic_data['Rainfall'] / safe_ref).round(4)
    synthetic_data.drop(columns=['RTR_ref','Rain_ref'], inplace=True)


[6/9] Tái tính Derived cols...


In [8]:
# ============================================================
# 7. ĐỒNG BỘ CATEGORICAL LABELS  — giữ nguyên từ v5
# ============================================================
print('[7/9] Đồng bộ Categorical labels...')

if 'Dominant_Soil_Texture' in synthetic_data.columns:
    conds = [(synthetic_data['Clay']>=40),(synthetic_data['Sand']>=50),(synthetic_data['Silt']>=50)]
    synthetic_data['Dominant_Soil_Texture'] = np.select(conds,['Clayey','Sandy','Silty'],default='Loamy')

if 'pH_Suitability' in synthetic_data.columns:
    conds_ph = [(synthetic_data['pH']<5.5),(synthetic_data['pH']>7.5)]
    synthetic_data['pH_Suitability'] = np.select(conds_ph,['Acidic','Alkaline'],default='Optimal')

# FIX v5: Water_Availability_Cat — 2 nhãn đúng
if 'Water_Availability_Cat' in synthetic_data.columns:
    WATER_THRESHOLD = 0.25
    synthetic_data['Water_Availability_Cat'] = np.where(
        synthetic_data['sm_rootzone'] > WATER_THRESHOLD, 'Optimal', 'Moderate'
    )


[7/9] Đồng bộ Categorical labels...


In [9]:
# ============================================================
# 7b. PROBABILISTIC YIELD CONSTRAINT — CẢI TIẾN #3 v6
# ============================================================
print('[7b/9] Probabilistic Yield constraint...')
synthetic_data = yield_probabilistic_constraint(
    synthetic_data, data, q_low=0.01, q_high=0.99
)


[7b/9] Probabilistic Yield constraint...
     Yield clamped: 189 / 20000 (0.9%)


In [10]:
# ============================================================
# 8. LƯU FILE
# ============================================================
print('[8/9] Lưu file...')

orig_cols = [c for c in data.columns if c in synthetic_data.columns]
extra     = [c for c in synthetic_data.columns if c not in orig_cols]
synthetic_data = synthetic_data[orig_cols + extra]

output_file = 'Agri_Data_SMOTE_Noise_v6.csv'
synthetic_data.to_csv(output_file, index=False)
print(f'     Đã lưu: {output_file}  ({len(synthetic_data):,} dòng)')


[8/9] Lưu file...
     Đã lưu: Agri_Data_SMOTE_Noise_v6.csv  (20,000 dòng)


In [11]:
# ============================================================
# 9. BÁO CÁO SO SÁNH TOÀN DIỆN
# ============================================================
print('[9/9] Đánh giá chất lượng...')

num_cols = [c for c in data.select_dtypes(include=np.number).columns
            if c in synthetic_data.columns]

# --- 1. Frobenius ---
co_s = data[num_cols].corr('spearman')
cn_s = synthetic_data[num_cols].corr('spearman')
co_p = data[num_cols].corr('pearson')
cn_p = synthetic_data[num_cols].corr('pearson')
frob_s = np.linalg.norm(co_s.values - cn_s.values, 'fro')
frob_p = np.linalg.norm(co_p.values - cn_p.values, 'fro')

# --- 2. PCA drift ---
Xo = data[num_cols].fillna(data[num_cols].median())
Xv = synthetic_data[num_cols].fillna(synthetic_data[num_cols].median())
sc2 = StandardScaler()
Xo_s, Xv_s = sc2.fit_transform(Xo), sc2.transform(Xv)
var_o = PCA(5).fit(Xo_s).explained_variance_ratio_
var_v = PCA(5).fit(Xv_s).explained_variance_ratio_

# --- 3. Wasserstein ---
w_total = 0
for col in num_cols:
    rng = data[col].max() - data[col].min()
    if rng > 0:
        w_total += wasserstein_distance(data[col], synthetic_data[col]) / rng

# --- 4. Adversarial Accuracy ---
df_r   = data[num_cols].assign(label=0)
df_s   = synthetic_data[num_cols].assign(label=1)
df_adv = pd.concat([df_r, df_s]).fillna(0)
clf    = GradientBoostingClassifier(n_estimators=100, random_state=SEED)
adv_scores = cross_val_score(clf, df_adv.drop('label',axis=1), df_adv['label'],
                              cv=5, scoring='accuracy')
adv_acc = adv_scores.mean()

# --- 5. Range violations ---
total_viol = sum(
    ((synthetic_data[c] < data[c].min()) | (synthetic_data[c] > data[c].max())).sum()
    for c in num_cols
)

# --- 6. Top 5 correlation drift ---
col_mae = {c:(co_s[c]-cn_s[c]).abs().mean() for c in num_cols}
top5    = sorted(col_mae.items(), key=lambda x:-x[1])[:5]

# --- 7. Consistency checks ---
y_ok = ((synthetic_data['Production']/synthetic_data['Area']).round(4)==synthetic_data['Yield']).mean()
wc   = synthetic_data['Wind_Max'].corr(synthetic_data['is_extreme_Wind_Max'])

print(f"""
{'='*65}
[THÀNH CÔNG] v6: {output_file}
{'='*65}
Tổng dòng             : {len(synthetic_data):,}

--- Distribution Fidelity ---
Wasserstein total     : {w_total:.4f}   (v5: 0.491  | target: <0.45)

--- Dependency Fidelity ---
Frobenius Spearman    : {frob_s:.4f}   (v5: 1.077  | v4: 0.840)
Frobenius Pearson     : {frob_p:.4f}   (v5: 1.462  | v4: 1.440)

--- Adversarial Accuracy ---
Adv. Accuracy         : {adv_acc:.4f}   (v5: 0.907  | target: ~0.75)
(0.50=indistinguishable, 1.00=completely different)

--- Data Quality ---
Range violations      : {total_viol}        (v5: 462)
""")

print('--- PCA Explained Variance ---')
for i,(a,b) in enumerate(zip(var_o,var_v)):
    flag = '✓' if abs(a-b)<0.02 else '△'
    print(f'  PC{i+1}: orig={a:.4f}  v6={b:.4f}  drift={abs(a-b):.4f} {flag}')

print('\n--- Top 5 Correlation Drift cols ---')
for c,v in top5:
    print(f'  {c:<35}: {v:.4f}')

print(f"""
--- Consistency Checks ---
Yield = Prod/Area     : {y_ok*100:.1f}%
Wind corr vs flag     : {wc:.3f}  (gốc: 0.390)
Wind_Max max          : {synthetic_data['Wind_Max'].max():.3f}  (gốc: 10.625)
Yield mean            : {synthetic_data['Yield'].mean():.4f}  (gốc: {data['Yield'].mean():.4f})
Yield std             : {synthetic_data['Yield'].std():.4f}  (gốc: {data['Yield'].std():.4f})
Heat_Stress max       : {synthetic_data['Heat_Stress_Days'].max():.1f}  (gốc: 42.5)
Water_Avail labels    : {dict(synthetic_data['Water_Availability_Cat'].value_counts().items())}
""")


[9/9] Đánh giá chất lượng...

[THÀNH CÔNG] v6: Agri_Data_SMOTE_Noise_v6.csv
Tổng dòng             : 20,000

--- Distribution Fidelity ---
Wasserstein total     : 0.4829   (v5: 0.491  | target: <0.45)

--- Dependency Fidelity ---
Frobenius Spearman    : 1.0065   (v5: 1.077  | v4: 0.840)
Frobenius Pearson     : 1.1897   (v5: 1.462  | v4: 1.440)

--- Adversarial Accuracy ---
Adv. Accuracy         : 0.8290   (v5: 0.907  | target: ~0.75)
(0.50=indistinguishable, 1.00=completely different)

--- Data Quality ---
Range violations      : 364        (v5: 462)

--- PCA Explained Variance ---
  PC1: orig=0.1723  v6=0.1844  drift=0.0121 ✓
  PC2: orig=0.1443  v6=0.1566  drift=0.0124 ✓
  PC3: orig=0.0886  v6=0.0850  drift=0.0036 ✓
  PC4: orig=0.0820  v6=0.0801  drift=0.0019 ✓
  PC5: orig=0.0653  v6=0.0747  drift=0.0094 ✓

--- Top 5 Correlation Drift cols ---
  Heat_Stress_Days                   : 0.0472
  Moisture_Ratio                     : 0.0367
  NDVI_Season_CV                     : 0.0304
  Bulk